# Chapter 10 Your data under a different lens: Window functions
we will disucss
- Window functions and the kind of data transformation they enable
- Summarizing, ranking, and analyzing data using the different classes of window functions
- Building static, growing, and unbounded windows to your functions
- Apply UDF to windows as custom window functions

### How to use this notebook

This is the in-class version. The narration and the working code are here; the
window piece each cell is teaching has been replaced by `___`.

- Fill in every `___` and run the cell. If it errors, read the message before
  changing anything — it usually names the column or function it could not find.
- `F.` is `pyspark.sql.functions` and `Window` is `pyspark.sql.window.Window`.
  `help(F.rank)` or `help(Window.rowsBetween)` explains one.
- Every window has up to three parts: how the data is split (`partitionBy`), how
  each split is ordered (`orderBy`), and how far the frame reaches
  (`rowsBetween` / `rangeBetween`). When a blank stumps you, ask which part it is.
- The five **Practice** questions at the end have no code at all. Write them
  yourself.
- Change `account` in the setup cell to your own Bryant username before you run
  anything.


In [1]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql.utils import AnalysisException
import pyspark.sql.functions as F
import pyspark.sql.types as T
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pyspark.sql.window import Window

import warnings
warnings.filterwarnings('ignore')

# change the account name to your email account
account='sli'

# define a root path to access the data in the DataAnalysisWithPythonAndPySpark
data_path='/net/clusterhn/home/'+account+'/isa460/data/'

# check if the Spark session is active. If it is activate, close it

try:
    if spark:
        spark.stop()
except:
    pass    

spark = (SparkSession.builder.appName("Window Functions")
        .config("spark.port.maxRetries", "100")
        .config("spark.sql.mapKeyDedupPolicy", "LAST_WIN")  # This configuration allow the duplicate keys in the map data type.
         .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
        .config("spark.driver.memory", "8g")
        .config("spark.driver.executor","8g")
        .getOrCreate())

# confiture the log level (defaulty is WARN)
spark.sparkContext.setLogLevel('ERROR')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/04 18:17:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/04 18:17:55 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/10/04 18:17:55 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


# Weather data

The data comes from NOAA's National Centers for Environmental Information (NCEI). It has already been downloaded, cleaned and saved as a parquet file, so you read it directly.

## Boston Logan Weather Data: Column Definitions

Source: NOAA NCEI Global Surface Summary of the Day (GSOD), station 72509014739 (Boston Logan International Airport), 2010–2025.

| Column | Definition | Unit |
|---|---|---|
| `date` | Observation date | YYYY-MM-DD |
| `avg_temp_f` | Mean temperature for the day | °F |
| `max_temp_f` | Highest temperature reported during the day | °F |
| `min_temp_f` | Lowest temperature reported during the day | °F |
| `precip_in` | Total precipitation (rain plus melted snow) | inches |
| `snow_depth_in` | Snow depth on the ground, from the last report of the day | inches |
| `wind_knots` | Mean wind speed for the day | knots (1 knot ≈ 1.15 mph) |
| `gust_knots` | Maximum wind gust reported | knots |
| `visibility_mi` | Mean visibility for the day | miles |
| `weather_flags` | 6-digit indicator (FRSHTT) for Fog, Rain/drizzle, Snow/ice pellets, Hail, Thunder, Tornado; 1 = occurred, 0 = did not | code |
| `year` | Year extracted from `date` | integer |
| `month` | Month extracted from `date` | 1–12 |
| `day` | Day of the month extracted from `date` | 1–31 |
| `year_month` | Year and month of `date` | YYYY-MM |
| `fog` … `tornado` | One column per `weather_flags` digit | 1 = occurred, 0 = did not |

**Notes**
- Missing-value codes (9999.9, 999.9, 99.99) were replaced with `NaN`.
- Max/min temperatures may include part of the previous day.
- A precipitation value of 0.00 can mean either no rain or no report.
- `weather_flags` is read as a string so leading zeros are kept (e.g., `010010` = rain and thunder).

# load data

In [2]:
# read the prepared weather data

df=spark.read.parquet(data_path+'boston_logan_weather_2010_2025.parquet')

df.printSchema()

df.show(5)

root
 |-- date: date (nullable = true)
 |-- avg_temp_f: double (nullable = true)
 |-- max_temp_f: double (nullable = true)
 |-- min_temp_f: double (nullable = true)
 |-- precip_in: double (nullable = true)
 |-- snow_depth_in: string (nullable = true)
 |-- wind_knots: double (nullable = true)
 |-- gust_knots: double (nullable = true)
 |-- visibility_mi: double (nullable = true)
 |-- weather_flags: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- fog: integer (nullable = true)
 |-- rain: integer (nullable = true)
 |-- snow: integer (nullable = true)
 |-- hail: integer (nullable = true)
 |-- thunder: integer (nullable = true)
 |-- tornado: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- year_month: string (nullable = true)

+----------+----------+----------+----------+---------+-------------+----------+----------+-------------+-------------+----+-----+---+----+----+----+-------+-------+---+----------+
|      date|

# Display average temperature by year

In [ ]:
df1=df.groupBy('year').agg(F.avg('avg_temp_f').alias('avg_temp')).orderBy('year')

df1.show()

In [ ]:
# visualize the result

result=df1.toPandas()

plt.figure(figsize=(14, 6))

sns.lineplot(data=result, x='year', y='avg_temp')

plt.title('Yearly Average Temperture in Boston Logan Between 2010 and 2025');

# Display daily temperature 

In [ ]:
result=df.toPandas()

result.head()

In [ ]:
plt.figure(figsize=(14, 6))

sns.lineplot(data=result, x='date', y='avg_temp_f')

plt.title('Daily Temperture in Boston Logan Between 2010 and 2025')

## Identify the coldest day of each year

In [ ]:
# A window spec that splits the data by year, and the method that runs
# an aggregate over that window instead of collapsing the rows.

from pyspark.sql.window import Window

windowSpec=Window.___('year')

result=df.withColumn('coldestDay', F.min('avg_temp_f').___(windowSpec))

result2=result.where('avg_temp_f=coldestDay')

result2.show()

## Identify the hottest day of each year

In [ ]:
windowSpec=Window.partitionBy('___')

result=df.withColumn('hottestDay', F.___('avg_temp_f').over(windowSpec))

result2=result.where('avg_temp_f=hottestDay')

result2.show()

# Ranking functions

This section covers ranking functions: 
- nonconsecutive ranks with rank()
- consecutive ranks with dense_rank()
- percentile ranks with percent_rank()
- tiles with ntile(), 
- finally a bare row number with row_number()

Ranking functions are used for getting the top (or bottom) record for each window partition, or, more generally, for getting an order according to some column’s value.

## Identify the top 3 hottest days per year

In [ ]:
# Hottest first, so the order is descending. Use the ranking function
# that skips numbers after a tie (1, 1, 3).

windowSpec=Window.partitionBy('year').orderBy(F.___('avg_temp_f'))

result=df.withColumn('rank', F.___().over(windowSpec)).filter(F.col('rank')<=3)

result.show()

## Identify the top 5% of the hottest day per year

In [ ]:
# The ranking function that returns a value between 0 and 1, and the
# cut-off that keeps the top 5%.

windowSpec=Window.partitionBy('year').orderBy('avg_temp_f')

result=df.withColumn('percent_rank', F.___().over(windowSpec)).where('percent_rank>=___')
result.show()

### Split the temp per year into 10 equal buckets (decile)

In [ ]:
# The function that splits each year into a given number of equal
# buckets. You want deciles.

windowSpec=Window.partitionBy('year').orderBy('avg_temp_f')

result=df.withColumn('decile', F.___(___).over(windowSpec)).groupBy('year', 'decile').count().orderBy('year', 'decile')
result.show()

In [ ]:
# check the temp in decile (10% of the coldest temperature in each year)

windowSpec=Window.partitionBy('year').orderBy('avg_temp_f')

df.withColumn('decile', F.ntile(10).over(windowSpec)).where('decile=___').show()

## Add a row number to your data frame, ignore tie

In [ ]:
# A plain 1, 2, 3, ... numbering that ignores ties.

windowSpec=Window.partitionBy('year').orderBy('avg_temp_f')

df.withColumn('row_number', F.___().over(windowSpec)).show()

## Access the records before or after using lag() and lead()

## Display average daily temp change by month

In [ ]:
# Compare each day with the day before it, inside the same month.
# Fill in the partition column, the order column, and the function
# that reads the PREVIOUS row's value.

windowSpec=Window.partitionBy('___').orderBy('___')

df2=df.select('year_month', 'day', 'avg_temp_f', F.___('avg_temp_f').over(windowSpec).alias('pre_temp'))

df2.show()

df3=df2.withColumn('daily_temp_change', F.abs(F.col('avg_temp_f')-F.col('pre_temp')))

df3.show()

df4=df3.groupBy('year_month').agg(F.round(F.avg('daily_temp_change'), 2).alias('avgTempChange'))

df4.show()

#### Visualize the result

In [ ]:
result=df4.toPandas()

result.head()

In [ ]:
plt.figure(figsize=(14, 6))

sns.lineplot(data=result, x='year_month', y='avgTempChange')

#### fix year_month display

##### Option 1: convert year_month to date format

In [ ]:
result.dtypes

In [ ]:
# convert year_month to Date

result['year_month'] = pd.to_datetime(result['year_month'], format='%Y-%m')

result.dtypes

In [ ]:
plt.figure(figsize=(14, 6))

sns.lineplot(data=result, x='year_month', y='avgTempChange')

##### keep year_month as string, show every sixth label

In [ ]:
result=df4.toPandas()

plt.figure(figsize=(14, 6))

ax = sns.lineplot(data=result, x='year_month', y='avgTempChange')

# Show every sixth label
labels = result['year_month'].tolist()
ax.set_xticks(range(0, len(labels), 6))
ax.set_xticklabels(labels[::6], rotation=45, ha='right')

plt.tight_layout()
plt.show()

#### display avg, max and min temp change by month

In [ ]:
df3.show()

In [ ]:
df4=df3.groupBy('year_month').agg(F.round(F.avg('daily_temp_change'),2).alias('avgTempChange')
                                , F.round(F.min('daily_temp_change'),2).alias('minTempChange')
                               , F.round(F.max('daily_temp_change'),2).alias('maxTempChange')
                               )
df4.show()

##### Visualize the result

In [ ]:
result=df4.toPandas()

result.head()

In [ ]:
result2 = pd.melt(result, id_vars=['year_month'], value_vars=['avgTempChange', 'maxTempChange', 'minTempChange'], 
                    var_name='temp_stat', value_name='temperature')

result2.head()

In [ ]:
# convert year_month to date format
result2['year_month'] = pd.to_datetime(result2['year_month'], format='%Y-%m')

In [ ]:
plt.figure(figsize=(12, 8))

sns.lineplot(data=result2, x='year_month', y='temperature', hue='temp_stat')

plt.title('Monthly Temperature Changge in Boston betwene 2010 and 2025', size=14)
plt.xlabel('year-month')
plt.ylabel('Taemperature Change')

## Spark also provides the rowsBetween() and rangeBetween() methods to create window frame boundaries.

### Display three days moving average temp for each month

In [ ]:
df.columns

In [ ]:
# A frame of the current row and the two rows before it. Positions are
# relative to the current row, which is 0.

windowSpec=Window.partitionBy('year_month').orderBy('day').rowsBetween(___, ___)

df1=df.withColumn('3_day_moving_avg', F.avg('avg_temp_f').over(windowSpec))

df1.show()

# Summary

- Window functions are functions that are applied over a portion of a data frame called a window frame. They can perform aggregation, ranking, or analytical operations. A window function will return the data frame with the same number of records, unlike its siblings the groupby-aggregate operation and the group map UDF.
- A window frame is defined through a window spec. A window spec mandates how the data frame is split (partitionBy()), how it’s ordered (orderBy()), and how it’s portioned (rowsBetween()/rangeBetween()).
- By default, an unordered window frame will be unbounded, meaning that the window frame will be equal to the window partition for every record. An ordered window frame will grow to the left, meaning that each record will have a window frame ranging from the first record in the window partition to the current record.
- A window can be bounded by row, meaning that the records included in the window frame are tied to the row boundaries passed as parameters (with the range boundaries added to the row number of the current row), or by range, meaning that the records included in the window frame depend on the value of the current row (with the range boundaries added to the value).

# Practice: Window functions

Use the weather data frame `df` loaded above. Every question can be answered with a
window function. Work in the empty cell under each question, and check each result
with `show()` before moving on.


## Question 1: Unusually warm days

For every day, compute the average temperature of the month (`year_month`) that the
day belongs to, and how many degrees the day's average temperature was above that
monthly average. Every row of the data should be kept.

Show the 10 days that were furthest above their month's average, with the date, the
day's average temperature, the month's average (rounded to 1 decimal), and the
difference (rounded to 1 decimal).


## Question 2: Record-warm months

Compute the average temperature for every year and month. Then, for each calendar
month (January through December), find the year in which that month was the warmest.

Show one row per month, ordered by month, with the year and that month's average
temperature (rounded to 1 decimal).


## Question 3: Sharpest cold snap

For each year, find the day with the largest one-day drop in average temperature,
compared with the previous day in the data.

Show one row per year with the year, the date, the previous day's temperature, the
day's temperature, and the size of the drop (rounded to 1 decimal).


## Question 4: Wettest week

For each day, compute the total precipitation over that day and the six records
before it within the same year. (A few days are missing from the data, so count six
previous records rather than six calendar days.)

Show the five dates with the largest seven-day totals, with that day's precipitation
and the seven-day total (rounded to 2 decimals).


## Question 5: Reaching 20 inches of precipitation

For each year, compute a running (year-to-date) total of precipitation that starts on
the first day of the year and adds each day in date order.

Then find the first date in each year on which the running total reached 20 inches or
more. Show one row per year, ordered by year.
